# A Dose-Response Evaluation of Explanation Metrics under Controlled Shortcut Reliance in CNN Image Classifiers

## Amendment A3, session 4: SD4 confirmatory set and ROAD on the GPU models

Pre-registered in `docs/PREDICTIONS_amendments.md` (A3) and `docs/PREDICTIONS_sd3_sd4.md`
(P10–P13), timestamped on Zenodo (10.5281/zenodo.24416017). This notebook trains and scores the 18 SD4 confirmatory models (metrics, oracle maps, controls, behaviour labels, exact ROAD) and scores the 69 registered `cifar10_resnet18`, `pet_resnet18` and `pet_resnet18_pretrained` patch models with exact ROAD.

Run only after the dated SD4 completion note (roadmap T4.3) is committed and uploaded
to Zenodo, and `A3_SD4_RULE` in `dose_response.py` is set from it (rebuild the bundle).
Fill `SD4_DOSES` in the parameters cell with the note's final doses.

**+ Add Input**: the A3 bundle dataset (code and data); the A3 models dataset (`xai_a3models_bundle.zip`: the 69 checkpoints); to resume, the Output of an earlier run of this notebook (`a3_session4*_results.zip`): finished models and ROAD files are skipped.

Before running: **Settings → Accelerator → GPU T4 x2**, and **Internet on**.
Then **Save Version → Save & Run All (Commit)**. The results are packed into
`a3_session4_results.zip` in the notebook's Output.

In [ ]:
# Parameters: the final SD4 doses of the dated SD4 completion note (T4.3,
# docs/PREDICTIONS_amendments.md): one of the two A3_SD4_DOSE_SETS lists.
SD4_DOSES = []
SEEDS = [42, 1, 2]  # confirmatory seeds; seed 7 is the SD4 pilot seed and is never used here
assert SD4_DOSES, 'Fill SD4_DOSES from the SD4 completion note first.'
assert 7 not in SEEDS, 'Seed 7 is reserved for the SD4 pilots.'

In [ ]:
# 1. Copy the uploaded project into the writable working folder
import glob, os, shutil, subprocess, sys
found = glob.glob('/kaggle/input/**/dose_response.py', recursive=True)
assert found, 'Project files not found: add your uploaded dataset with "+ Add Input".'
src = os.path.dirname(found[0])
WORK = '/kaggle/working/project'
if not os.path.exists(WORK):
    shutil.copytree(src, WORK)
os.chdir(WORK)
print('project copied from', src)
print(sorted(os.listdir(WORK)))

# Resume: unpack an attached earlier Output of this notebook, then copy every attached
# checkpoint and per-model file not yet present (Kaggle may show zips unpacked).
import zipfile
for archive in glob.glob('/kaggle/input/**/a3_session4*_results.zip', recursive=True):
    with zipfile.ZipFile(archive) as z:
        z.extractall(WORK)
    print('extracted', archive)
for folder in ('checkpoint/dose', 'results/dose_response/per_model'):
    os.makedirs(folder, exist_ok=True)
    copied = 0
    for f in glob.glob(f'/kaggle/input/**/{folder}/*', recursive=True):
        target = os.path.join(folder, os.path.basename(f))
        if os.path.isfile(f) and not os.path.exists(target):
            shutil.copy(f, target)
            copied += 1
    print(copied, 'files copied into', folder)
sys.path.insert(0, WORK)
import dose_response
SD4 = [('pet_resnet18_pretrained', 'watermark', float(d), s) for d in SD4_DOSES for s in SEEDS]
assert any(list(map(float, SD4_DOSES)) == list(doses) for doses in dose_response.A3_SD4_DOSE_SETS.values()), (
    'SD4_DOSES must be one of the A3 dose sets: ' + str(dose_response.A3_SD4_DOSE_SETS))
wrong = [dose_response.model_name(*c) for c in SD4
         if dose_response.model_set_of(dict(zip(('base', 'shortcut', 'dose', 'seed'), c))) != 'A3']
assert not wrong, ('not in model set A3 (set A3_SD4_RULE in dose_response.py from the completion note '
                   'and rebuild the bundle): ' + ', '.join(wrong))
print(len(SD4), 'SD4 models, model set A3')
ROAD_GPU_BASES = ['cifar10_resnet18', 'pet_resnet18', 'pet_resnet18_pretrained']
ROAD_GPU = sorted({dose_response.model_name(b, 'patch', d, s)
                   for design in (dose_response.ORIGINAL_DESIGN, dose_response.A2_DESIGN)
                   for b in ROAD_GPU_BASES if b in design for d in design[b][1] for s in design[b][2]})
assert len(ROAD_GPU) == 69, len(ROAD_GPU)
missing = [n for n in ROAD_GPU if not all(os.path.exists(f'checkpoint/dose/{n}{s}') for s in ('.pt', '_metrics.json'))]
assert not missing, (f'{len(missing)} ROAD checkpoints missing, e.g. {missing[0]}: attach the A3 models dataset '
                     '(make_bundle.ps1 -Dataset a3models).')
print('all 69 ROAD GPU models found')

In [ ]:
# 2. Install the one missing package, check the GPU and the data, get the ImageNet weights
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'captum==0.9.0'], check=True)
import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'No GPU: turn on Settings → Accelerator → GPU, then restart.'
for name in ('trainval.npz', 'test.npz'):
    assert os.path.exists(f'data/pet128/{name}'), f'data/pet128/{name} missing: rebuild the bundle with -Dataset a3.'
assert os.path.isdir('data/cifar-10-batches-py'), 'data/cifar-10-batches-py missing: rebuild the bundle with -Dataset a3.'
print('Pet and CIFAR-10 data found')
from torchvision.models import ResNet18_Weights, resnet18
try:
    resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)  # cached for the training cells
except Exception as error:
    raise RuntimeError('ImageNet weights download failed: turn Internet on (Settings → Internet).') from error
print('ImageNet ResNet-18 weights ready')

In [ ]:
# 3. SD4 confirmatory: train, score, controls, behaviour labels and exact ROAD.
#    Re-running skips finished models and files.
def run(*args):
    subprocess.run([sys.executable, '-u', '-W', 'ignore', 'dose_response.py', *args], check=True)

selection = ['--bases', 'pet_resnet18_pretrained', '--shortcuts', 'watermark', '--doses', *map(str, SD4_DOSES),
             '--seeds', *map(str, SEEDS)]
run('train', *selection)
run('evaluate', *selection)
run('placebo', *selection)
run('behaviour', '--selection-only', *selection)
run('road', '--selection-only', *selection)

In [ ]:
# 4. Exact ROAD of the registered GPU patch models (pilots are not scored).
#    Resumable: a model whose per_model/<name>_road.csv exists is skipped.
run('road', '--bases', *ROAD_GPU_BASES, '--shortcuts', 'patch')

In [ ]:
# 5. Pack the results to download: the SD4 checkpoints and per-model files, and
#    every per_model/*_road.csv (the attached GPU checkpoints are not repacked)
names = [dose_response.model_name(*c) for c in SD4]
files = sorted({f for n in names for f in glob.glob(f'checkpoint/dose/{n}.pt')
                 + glob.glob(f'checkpoint/dose/{n}_metrics.json') + glob.glob(f'results/dose_response/per_model/{n}_*.csv')}
               | set(glob.glob('results/dose_response/per_model/*_road.csv')))
out = '/kaggle/working/a3_session4_results.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, f)  # keeps checkpoint/... and results/... paths
print(len(files), 'files ->', out)
shutil.rmtree(WORK)  # keep only the zip in the Output